In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
import heapq
import warnings
warnings.filterwarnings('ignore')


In [ ]:

df = pd.read_csv("/Users/shuddhisharma/Downloads/Bengaluru_House_Data.csv")
df = df[['total_sqft', 'price']].dropna()

def convert_sqft(x):
    try:
        s = str(x).strip()
        if '-' in s:
            a, b = s.split('-')
            return (float(a) + float(b)) / 2
        return float(s)
    except Exception:
        return None

df['total_sqft'] = df['total_sqft'].apply(convert_sqft)
df = df.dropna()
df = df[df['total_sqft'] > 0]
df = df[df['price'] > 0]

print(f"Dataset loaded: {len(df)} records after cleaning")


In [ ]:
df['price_per_sqft'] = (df['price'] * 100000) / df['total_sqft']

Q1  = df['price_per_sqft'].quantile(0.25)
Q3  = df['price_per_sqft'].quantile(0.75)
IQR = Q3 - Q1
df  = df[
    (df['price_per_sqft'] >= Q1 - 1.5 * IQR) &
    (df['price_per_sqft'] <= Q3 + 1.5 * IQR)
]

sqft_Q1 = df['total_sqft'].quantile(0.05)
sqft_Q3 = df['total_sqft'].quantile(0.95)
df = df[(df['total_sqft'] >= sqft_Q1) & (df['total_sqft'] <= sqft_Q3)]

print(f"After outlier removal: {len(df)} records")

PLOT_SIZES = [500, 1000, 1500]

def categorize(size):
    if size < 800:
        return 500
    elif size < 1200:
        return 1000
    else:
        return 1500

df['plot_size'] = df['total_sqft'].apply(categorize)

price_per_sqft = df.groupby('plot_size')['price_per_sqft'].median().to_dict()
demand         = df['plot_size'].value_counts().to_dict()

print("\nMedian price per sqft (Rs):")
for size in PLOT_SIZES:
    print(f"  {size} sqft -> Rs {price_per_sqft[size]:,.0f}/sqft")


In [ ]:

def prim_mst_cost(points):
    n = len(points)
    if n < 2:
        return 0.0
    in_mst   = [False] * n
    min_dist = [float('inf')] * n
    min_dist[0] = 0.0
    total_cost = 0.0
    heap = [(0.0, 0)]
    while heap:
        d, u = heapq.heappop(heap)
        if in_mst[u]:
            continue
        in_mst[u]  = True
        total_cost += d
        for v in range(n):
            if not in_mst[v]:
                dx   = points[u][0] - points[v][0]
                dy   = points[u][1] - points[v][1]
                dist = (dx**2 + dy**2) ** 0.5
                if dist < min_dist[v]:
                    min_dist[v] = dist
                    heapq.heappush(heap, (dist, v))
    return total_cost

In [ ]:
# ── Constants ─────────────────────────────────────────────────────────────────
OPEN_SPACE_RATIO   = 0.30      # 30% reserved for setbacks / open space
COST_PER_M_ROAD    = 20_000    # Rs per metre — realistic Bengaluru residential road rate
SQFT_TO_M          = 0.3048    # conversion factor
LAND_COST_PER_SQFT = 3_000     # Rs/sqft — approximate Bengaluru land acquisition cost

# ── Greedy allocation ─────────────────────────────────────────────────────────
# Why greedy? Plot allocation is a variant of fractional knapsack:
# plots are divisible, so greedy (sort by score, fill in order) is optimal.
# Score = price_per_sqft × demand_share — balances market rate and demand.
def allocate_plots(total_land):
    buildable    = int(total_land * (1 - OPEN_SPACE_RATIO))
    total_demand = sum(demand.get(s, 1) for s in PLOT_SIZES)
    scores = {
        s: price_per_sqft[s] * (demand.get(s, 1) / total_demand)
        for s in PLOT_SIZES
    }
    ranked     = sorted(PLOT_SIZES, key=lambda s: scores[s], reverse=True)
    allocation = {}
    remaining  = buildable
    for size in ranked:
        count          = remaining // size
        allocation[size] = int(count)
        remaining      -= count * size
    return allocation, remaining

In [ ]:
# LAYOUT
SIZE_SIDE = {500: 22, 1000: 32, 1500: 39}
GUTTER    = 5

def build_layout(allocation):
    plots     = []
    x, y      = 0, 0
    row_max_h = 0
    MAX_WIDTH = 250

    for size in PLOT_SIZES:               # always iterate in fixed order
        count = allocation.get(size, 0)
        if count == 0:
            continue
        w = h = SIZE_SIDE[size]
        for _ in range(count):
            if x + w > MAX_WIDTH:
                x         = 0
                y        += row_max_h + GUTTER
                row_max_h = 0
            plots.append({'x': x, 'y': y, 'w': w, 'h': h, 'size': size})
            x        += w + GUTTER
            row_max_h = max(row_max_h, h)

    return plots

In [ ]:
# ── Simulation ────────────────────────────────────────────────────────────────
# ROI is calculated on land acquisition cost (the actual capital at risk),
# not on revenue — giving a realistic investment return figure.
# Note: this is a planning-phase simulator. It intentionally excludes permits,
# utilities, and marketing costs to focus on core algorithmic outputs.
def run_simulation(total_land):
    allocation, leftover = allocate_plots(total_land)
    plots     = build_layout(allocation)

    if not plots:
        raise ValueError(f"No plots generated for land={total_land} sqft")

    centroids = [(p['x'] + p['w'] / 2, p['y'] + p['h'] / 2) for p in plots]

    mst_feet   = prim_mst_cost(centroids)
    mst_metres = mst_feet * SQFT_TO_M
    road_cost  = mst_metres * COST_PER_M_ROAD

    total_revenue        = sum(
        size * price_per_sqft[size] * count
        for size, count in allocation.items()
    )
    land_acquisition_cost = total_land * LAND_COST_PER_SQFT
    profit                = total_revenue - road_cost - land_acquisition_cost
    roi                   = (profit / land_acquisition_cost * 100) if land_acquisition_cost > 0 else 0

    return {
        'total_land'   : total_land,
        'allocation'   : allocation,
        'leftover_sqft': leftover,
        'plots'        : plots,
        'num_plots'    : len(plots),
        'mst_metres'   : mst_metres,
        'revenue'      : total_revenue,
        'road_cost'    : road_cost,
        'land_cost'    : land_acquisition_cost,
        'profit'       : profit,
        'roi_pct'      : roi,
    }


In [ ]:
# ── Visualisation helpers ──────────────────────────────────────────────────────

COLOR_MAP = {500: '#4CAF50', 1000: '#2196F3', 1500: '#FF5722'}

def plot_layout(result, ax):
    plots = result['plots']
    if not plots:
        ax.text(0.5, 0.5, 'No plots generated',
                ha='center', va='center', transform=ax.transAxes)
        return

    for p in plots:
        color = COLOR_MAP.get(p['size'], '#999999')
        rect  = mpatches.FancyBboxPatch(
            (p['x'], p['y']), p['w'], p['h'],
            boxstyle="round,pad=0.5",
            facecolor=color,
            edgecolor='white', linewidth=0.8, alpha=0.85
        )
        ax.add_patch(rect)
        ax.text(
            p['x'] + p['w'] / 2,
            p['y'] + p['h'] / 2,
            str(p['size']),
            ha='center', va='center',
            fontsize=6, color='white', fontweight='bold'
        )

    # Force axis limits manually so patches are visible
    all_x = [p['x'] + p['w'] for p in plots]
    all_y = [p['y'] + p['h'] for p in plots]
    ax.set_xlim(-5, max(all_x) + 10)
    ax.set_ylim(-5, max(all_y) + 10)
    ax.set_aspect('equal')

    ax.set_title(
        f"Layout — {result['total_land']:,} sqft land\n"
        f"{result['num_plots']} plots | {result['leftover_sqft']} sqft unused",
        fontsize=10
    )
    ax.set_xlabel("feet")
    ax.set_ylabel("feet")

    handles = [
        mpatches.Patch(color=COLOR_MAP[s], label=f"{s} sqft")
        for s in PLOT_SIZES if result['allocation'].get(s, 0) > 0
    ]
    ax.legend(handles=handles, fontsize=8, loc='upper right')


def plot_financials(results, ax):
    labels    = [f"{r['total_land']:,}" for r in results]
    revenue   = [r['revenue']    / 1e7 for r in results]
    land_cost = [r['land_cost']  / 1e7 for r in results]
    road_cost = [r['road_cost']  / 1e7 for r in results]
    profits   = [r['profit']     / 1e7 for r in results]

    x     = np.arange(len(labels))
    width = 0.20

    ax.bar(x - 1.5*width, revenue,   width, label='Revenue',      color='#4CAF50', alpha=0.85)
    ax.bar(x - 0.5*width, land_cost, width, label='Land Cost',    color='#FF9800', alpha=0.85)
    ax.bar(x + 0.5*width, road_cost, width, label='Road Cost',    color='#F44336', alpha=0.85)
    ax.bar(x + 1.5*width, profits,   width, label='Net Profit',   color='#2196F3', alpha=0.85)

    for i, r in enumerate(results):
        ax.text(
            x[i] + 1.5*width, profits[i] + 0.01,
            f"ROI\n{r['roi_pct']:.1f}%",
            ha='center', va='bottom', fontsize=8,
            color='#2196F3', fontweight='bold'
        )

    ax.set_xticks(x)
    ax.set_xticklabels([f"{l} sqft" for l in labels])
    ax.set_ylabel("Rs Crore")
    ax.set_title("Financial Comparison Across Land Sizes")
    ax.legend()
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda v, _: f"Rs{v:.1f}Cr")
    )


def plot_price_distribution(ax):
    data   = [df[df['plot_size'] == s]['price_per_sqft'].values for s in PLOT_SIZES]
    bp     = ax.boxplot(data, patch_artist=True, notch=False)
    colors = [COLOR_MAP[s] for s in PLOT_SIZES]
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_xticklabels([f"{s} sqft" for s in PLOT_SIZES])
    ax.set_ylabel("Rs / sqft")
    ax.set_title("Price Distribution by Plot Size")
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda v, _: f"Rs{v:,.0f}")
    )


def plot_allocation_pie(results, axes):
    for ax, r in zip(axes, results):
        sizes  = [r['allocation'].get(s, 0) for s in PLOT_SIZES]
        labels = [f"{s} sqft\n({r['allocation'].get(s, 0)})" for s in PLOT_SIZES]
        colors = [COLOR_MAP[s] for s in PLOT_SIZES]
        filtered = [(s, l, c) for s, l, c in zip(sizes, labels, colors) if s > 0]
        if filtered:
            s_vals, l_vals, c_vals = zip(*filtered)
            ax.pie(s_vals, labels=l_vals, colors=c_vals,
                   autopct='%1.1f%%', startangle=140)
        ax.set_title(f"{r['total_land']:,} sqft land")

In [ ]:
land_sizes = [8000, 10000, 15000]
results    = []

print("\n" + "="*55)
print("  BENGALURU REAL ESTATE PLOT SIMULATION - RESULTS")
print("="*55)

for land in land_sizes:
    try:
        r = run_simulation(land)
        results.append(r)

        buildable = int(r['total_land'] * (1 - OPEN_SPACE_RATIO))

        print(f"\n{'─'*48}")
        print(f"  Land Size    : {r['total_land']:,} sqft")
        print(f"  Buildable    : {buildable:,} sqft  (30% setback)")
        print(f"  Leftover     : {r['leftover_sqft']:,} sqft")
        print(f"  Total Plots  : {r['num_plots']}")
        print(f"\n  Plot Allocation:")
        for size in PLOT_SIZES:
            count = r['allocation'].get(size, 0)
            pps   = price_per_sqft[size]
            print(f"    {size} sqft -> {count} plots  (Rs {pps:,.0f}/sqft)")
        print(f"\n  Road Length  : {r['mst_metres']:,.1f} m")
        print(f"  Revenue      : Rs {r['revenue']/1e7:.2f} Cr")
        print(f"  Land Cost    : Rs {r['land_cost']/1e7:.2f} Cr")
        print(f"  Road Cost    : Rs {r['road_cost']/1e7:.2f} Cr")
        print(f"  Profit       : Rs {r['profit']/1e7:.2f} Cr")
        print(f"  ROI          : {r['roi_pct']:.1f}%")

    except ValueError as e:
        print(f"\n[SKIP] Land={land}: {e}")

# Figure 1 — Plot Layouts
n = len(results)
fig1, axes1 = plt.subplots(1, n, figsize=(7 * n, 7))
if n == 1:
    axes1 = [axes1]
for ax, r in zip(axes1, results):
    plot_layout(r, ax)
fig1.suptitle("Plot Layouts (Scaled Rectangles)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Figure 2 — Financials
fig2, ax2 = plt.subplots(figsize=(10, 5))
plot_financials(results, ax2)
plt.tight_layout()
plt.show()

# Figure 3 — Price Distribution
fig3, ax3 = plt.subplots(figsize=(8, 5))
plot_price_distribution(ax3)
plt.tight_layout()
plt.show()

# Figure 4 — Allocation Pie Charts
fig4, axes4 = plt.subplots(1, n, figsize=(5 * n, 5))
if n == 1:
    axes4 = [axes4]
plot_allocation_pie(results, axes4)
fig4.suptitle("Plot Mix by Land Size", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
summary = pd.DataFrame([{
    'Land (sqft)'      : r['total_land'],
    'Total Plots'      : r['num_plots'],
    'Road Length (m)'  : round(r['mst_metres'], 1),
    'Revenue (Rs Cr)'  : round(r['revenue'] / 1e7, 2),
    'Land Cost (Rs Cr)': round(r['land_cost'] / 1e7, 2),
    'Road Cost (Rs Cr)': round(r['road_cost'] / 1e7, 2),
    'Profit (Rs Cr)'   : round(r['profit'] / 1e7, 2),
    'ROI %'            : round(r['roi_pct'], 1),
} for r in results])

print("\n===== SUMMARY TABLE =====")
print(summary.to_string(index=False))